# 📏 Chapter 4: Distance Metrics and Nearest Neighbor
**Referensi Buku:** *scikit-learn Cookbook, Third Edition*

---
## 1. Pendahuluan
Algoritma *Nearest Neighbor* didasarkan pada prinsip sederhana: benda-benda yang serupa akan berada berdekatan satu sama lain di ruang fitur. Kunci utama dari algoritma ini adalah bagaimana kita mendefinisikan dan menghitung "Jarak" (Distance) tersebut.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split

%matplotlib inline
np.random.seed(42)

## 2. Metrik Jarak (Distance Metrics)
Sebelum menggunakan model K-NN, kita perlu memahami cara menghitung jarak. `scikit-learn` menyediakan `pairwise_distances` untuk menghitung jarak antar titik data secara efisien dengan berbagai metrik.

In [ ]:
from sklearn.metrics import pairwise_distances

# Kita buat dua titik sederhana di ruang 2D
titik_A = [[0, 0]]
titik_B = [[3, 4]]

# 1. Euclidean Distance (Garis lurus, rumus Phytagoras)
euclidean = pairwise_distances(titik_A, titik_B, metric='euclidean')
print(f"Jarak Euclidean (Garis lurus): {euclidean[0][0]}") # Hasilnya pasti 5

# 2. Manhattan Distance (Pergerakan kotak/blok kota)
manhattan = pairwise_distances(titik_A, titik_B, metric='manhattan')
print(f"Jarak Manhattan (3 ke kanan + 4 ke atas): {manhattan[0][0]}") # Hasilnya pasti 7

## 3. K-Nearest Neighbors Classifier (Klasifikasi)
Kita akan menggunakan K-NN untuk tugas klasifikasi. Parameter penting di sini adalah `n_neighbors` (jumlah tetangga) dan `p` (p=2 berarti Euclidean, p=1 berarti Manhattan).

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Membuat dataset klasifikasi buatan
X_clf, y_clf = make_classification(n_samples=300, n_features=2, n_redundant=0, n_clusters_per_class=1, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.3)

# Menggunakan 5 tetangga terdekat dengan jarak Euclidean (p=2)
knn_clf = KNeighborsClassifier(n_neighbors=5, p=2)
knn_clf.fit(X_train_c, y_train_c)

y_pred_c = knn_clf.predict(X_test_c)
print(f"Akurasi KNN Classifier: {accuracy_score(y_test_c, y_pred_c):.2f}")

# Fungsi Visualisasi Batas Keputusan
def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05), np.arange(y_min, y_max, 0.05))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='winter')
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='winter')
    plt.title(title)
    plt.show()

plot_decision_boundary(knn_clf, X_test_c, y_test_c, 'Batas Keputusan KNN (K=5)')

## 4. K-Nearest Neighbors Regressor (Regresi)
KNN juga dapat digunakan untuk memprediksi nilai numerik berkesinambungan. Hasil prediksinya adalah rata-rata dari target $K$ tetangganya.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

# Dataset regresi 1 Dimensi agar mudah divisualisasikan
X_reg = np.sort(5 * np.random.rand(80, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + np.random.normal(0, 0.1, 80)

knn_reg = KNeighborsRegressor(n_neighbors=5)
knn_reg.fit(X_reg, y_reg)

# Memprediksi garis kontinu
T = np.linspace(0, 5, 500)[:, np.newaxis]
y_pred_reg = knn_reg.predict(T)

plt.figure(figsize=(7, 5))
plt.scatter(X_reg, y_reg, color='darkorange', label='Data Asli')
plt.plot(T, y_pred_reg, color='navy', linewidth=2, label='Prediksi KNN Regresi')
plt.title('K-Nearest Neighbors Regressor (K=5)')
plt.legend()
plt.show()

## 5. Radius Neighbors Classifier
Pada data yang kepadatannya tidak merata, memaksa model mencari $K$ tetangga bisa berbahaya (karena tetangga ke-5 bisa saja berada sangat jauh). Solusinya adalah mencari semua tetangga yang berada dalam **radius** tertentu.

In [ ]:
from sklearn.neighbors import RadiusNeighborsClassifier

# Mencari semua tetangga dalam radius 1.5 satuan jarak
radius_clf = RadiusNeighborsClassifier(radius=1.5, outlier_label='most_frequent')
radius_clf.fit(X_train_c, y_train_c)

y_pred_radius = radius_clf.predict(X_test_c)
print(f"Akurasi Radius Neighbors (Radius=1.5): {accuracy_score(y_test_c, y_pred_radius):.2f}")

plot_decision_boundary(radius_clf, X_test_c, y_test_c, 'Batas Keputusan Radius Neighbors')